<!-- DOC-BANNER -->
# Fetching Data using Web Scraping (Day 18)

> **Playlist:** CampusX Day 18 *Fetching data using Web Scraping*.
> **Purpose:** Scrape company names/ratings/counts from `ambitionbox.com/...?page=4` with `requests` + `BeautifulSoup(lxml)`. Sends a Chrome `User-Agent` so the site treats the script as a browser.
> **Prerequisites:** HTML tags/classes basics, `requests.get().text`.
> **After studying:** you can fetch HTML → `BeautifulSoup(..., 'lxml')` → `find_all`/`find(...).get_text()` for one page.


## Web-scraping demo: company list from HTML

When no API exists, we fetch the page HTML and pull fields out of it with BeautifulSoup.


### 2. Parse and select
`BeautifulSoup(html, 'lxml')` builds a parse tree. `find_all('h2')` / rating `div`s / count `span`s extract the three fields; per-card `card.find(...).get_text()` narrows to one company (`companyCardWrapper` divs). Requires `pip install lxml` (done inside the notebook — prefer installing in the environment instead).

### 1. Look like a browser
`headers={'User-Agent': 'Mozilla/5.0 ... Chrome/142'}` avoids naive bot blocks. `requests.get(url, headers=headers).text` gives raw HTML.

In [ ]:
import pandas as pd
import requests 
from bs4  import BeautifulSoup 
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36"
}
url = "https://www.ambitionbox.com/list-of-companies?campaign=desktop_nav%3E%3Fpage%20%3D4"
webpage = requests.get(url, headers= headers).text

webpage



In [ ]:
pip install lxml


### Send browser-like headers so the request looks human

Sites often block plain scripts; a Chrome `User-Agent` header avoids the naive bot filter.


In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.ambitionbox.com/list-of-companies?campaign=desktop_nav%3E%3Fpage%20%3D4"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)

webpage = response.text

soup = BeautifulSoup(webpage, "lxml")




In [ ]:
for  i in soup.find_all('h2'):
    print( i.text.strip())

In [ ]:
print("rating_text" in webpage)

In [ ]:

ratings = soup.find_all("div", class_="rating_text")

for rating in ratings:
    print(rating.text.strip())

### Example: the rating-count element we extract

The count (e.g. `(1.2L)`) lives in a `span` like this — `find_all` + `get_text()` pulls it out:

```html
<span class="companyCardWrapper__companyRatingCount">(1.2L)</span>
```


In [ ]:
rating_count  = soup.find_all("span" , class_= "companyCardWrapper__companyRatingCount")
for cnt in rating_count : 
    print(cnt.text.strip())

In [ ]:
for rating , count in zip(ratings, rating_count) : 
    print(rating.text.strip() , count.text.strip())
        

### Putting it together: parse one company card

Narrow from the whole page to a single `companyCardWrapper` card, then read its fields.


In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.ambitionbox.com/list-of-companies?campaign=desktop_nav%3E%3Fpage%20%3D4"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)

webpage = response.text

soup = BeautifulSoup(webpage, "lxml")




In [ ]:
company_cards = soup.find_all(
    "div",
    class_="companyCardWrapper"
)

print(len(company_cards))


In [ ]:
company_cards = soup.find_all(
    "div",
    class_="companyCardWrapper"
)

(company_cards)

### One card at a time: extract name, rating and count

`card.find(...).get_text()` reads one field; repeat per card (guard against `None` when a field is missing).


In [ ]:
for card in company_cards:

    company_name = card.find(
        "h2",
        class_="companyCardWrapper__companyName"
    )

    rating = card.find(
        "div",
        class_="rating_text"
    )

    rating_count = card.find(
        "span",
        class_="companyCardWrapper__companyRatingCount"
    )

    print(company_name.get_text(strip=True))
    print(rating.get_text(strip=True))
    print(rating_count.get_text(strip=True))

<!-- EXPERIMENT -->
### Experiment: the missing-field crash
Pick one card and call `card.find("nonexistent-tag")` — predict what it returns, then watch `.get_text()` fail on it. Write the one-line `None` guard (`el.get_text() if el else None`) and confirm the loop survives. That guard is the difference between a demo and a scraper.

<!-- DOC-BANNER -->
## Key takeaways (Day 18)
- Scraping pattern: headers → GET → parse → find → text. Works for one page only here.
- **Unfinished:** no DataFrame/CSV export, duplicated request blocks, no `None` guard (`card.find()` can return `None` → `.get_text()` crashes), no multi-page loop.
- **Common mistakes:** scraping without checking robots/ToS; brittle class names; no delays/retries.
- **ML connection:** last-resort data collection when no API/dataset exists — scraped tables still need cleaning before EDA.
